# 03 — Parameterization results (per-HRU)

Choropleths + distributions of every merged per-HRU parameter in `{data_root}/{FABRIC}/params/merged/`, joined to the fabric geometry by `id_feature`. Continuous params get a map + histogram; categorical params (`soils`, `cov_type`) get a map + class legend. Missing CSVs are skipped.

> **Run this in JupyterHub on a compute node with enough `--mem`.** Rendering a
> full fabric (especially CONUS `gfv2`, ~361k HRUs) loads large geometries and
> rasters; the login node will run out of memory. Pick the fabric with the
> `FABRIC` env var (or edit the first code cell). Set `SAVE_FIGURES=1` (or
> `viz.SAVE_FIGURES = True`) to write PNGs into `docs/figures/{FABRIC}/`, or
> batch-generate them with `python scripts/render_figures.py --fabric {FABRIC}`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from gfv2_params import viz
from gfv2_params.config import load_base_config

# Fabric is selected by the FABRIC env var (set it in your JupyterHub session),
# or edit the default here. SAVE_FIGURES=1 writes PNGs to docs/figures/{FABRIC}/.
FABRIC = os.environ.get("FABRIC", "oregon")
viz.FABRIC = FABRIC
viz.SAVE_FIGURES = os.environ.get("SAVE_FIGURES", "0") == "1"

cfg = load_base_config(fabric=FABRIC)
DATA_ROOT = Path(cfg["data_root"])
HRU_GPKG = Path(cfg["hru_gpkg"])
HRU_LAYER = cfg["hru_layer"]
ID_FEATURE = cfg["id_feature"]

print(f"FABRIC       = {FABRIC}")
print(f"SAVE_FIGURES = {viz.SAVE_FIGURES}  ->  docs/figures/{FABRIC}/")
print(f"data_root    = {DATA_ROOT}")
print(f"hru_gpkg     = {HRU_GPKG} (layer={HRU_LAYER}, id={ID_FEATURE})")

## Load fabric geometry once + locate the merged params

In [ ]:
import geopandas as gpd
import pandas as pd

fabric_gdf = gpd.read_file(HRU_GPKG, layer=HRU_LAYER)[[ID_FEATURE, "geometry"]]
PARAMS_DIR = DATA_ROOT / FABRIC / "params" / "merged"
print(f"{len(fabric_gdf):,} HRUs;  params dir = {PARAMS_DIR}")

## Per-parameter maps

In [ ]:
for p in viz.param_inventory():
    csv_path = PARAMS_DIR / p.csv_name
    if not csv_path.exists():
        print(f"skip {p.name}: missing {p.csv_name}")
        continue
    gdf = viz.load_param(PARAMS_DIR, p.csv_name, fabric_gdf, ID_FEATURE)
    title = f"{p.name}  [{FABRIC}]"
    if p.kind == "categorical":
        fig = viz.map_categorical(gdf, p.column, title)
    else:
        fig = viz.map_continuous(gdf, p.column, title, units=p.units, cmap=p.cmap)
    viz.save_figure(fig, f"param_{p.column}")
    plt.show()

## Depression-storage ratio summary

Quick distribution table for the clamped `[0, 1]` depstor ratios (`carea_max`, `smidx_coef`, `sro_to_dprst_perv/imperv`, `hru_percent_imperv`, `dprst_frac`).

In [ ]:
ratio_cols = {"carea_max", "smidx_coef", "sro_to_dprst_perv",
              "sro_to_dprst_imperv", "hru_percent_imperv", "dprst_frac"}
rows = []
for p in viz.param_inventory():
    if p.column not in ratio_cols:
        continue
    csv_path = PARAMS_DIR / p.csv_name
    if not csv_path.exists():
        continue
    s = pd.read_csv(csv_path)[p.column].dropna()
    rows.append({"param": p.column, "n": len(s), "mean": s.mean(),
                 "median": s.median(), "min": s.min(), "max": s.max(),
                 "frac_zero": float((s == 0).mean()),
                 "frac_one": float((s >= 1).mean())})
pd.DataFrame(rows).set_index("param")